#Put your Google Colab link here:
*your link here*

## Important notice: any use of generative AI for completing the assignment is strictly prohibited.

## Get access to a GPU:
To gain access to the GPUs on Colab, navigate to the `Runtime` tab above and select `Change runtime type`.

In [ ]:
# use if working in colab
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)


## Import packages

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

### Warning: to ensure the reproducibility of your results and to achieve the full grade, do not change or remove RANDOM_STATE variables and setting random seed statements. If you remove or change them, you may not get the full grade.

In [ ]:
random_state = 5

In [ ]:
!pip install ucimlrepo

## Part 1: Data Loading and Preprocessing (3 points)

**Dataset: Cardiotocography (CTG)**
The Cardiotocography dataset contains 2,126 fetal heart rate (FHR) monitoring records from the UCI ML Repository (ID 193). Each record has 20 continuous clinical measurements (e.g. FHR baseline, accelerations, decelerations, variability) and one categorical feature — **Tendency** (FHR histogram asymmetry: left-skewed = −1, symmetric = 0, right-skewed = +1). The prediction target is the **NSP** (Neonatal State of Pregnancy) class: Normal (0), Suspect (1), or Pathologic (2).

Objectives:
- Load and understand the dataset structure
- Implement data reduction strategy
- Prepare training/validation/test splits


### Part 1.1: Data Loading (1 point)
Load the Cardiotocography dataset from UCI and check basic statistics.

The preprocessing code below downloads the CTG dataset, engineers the feature vector (20 continuous features + 3-category one-hot encoding of Tendency), performs an 60/20/20 train/validation/test split, and saves the result to your Google Drive.

- HINT: Use the provided dataloader function to load the saved .npz file

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from ucimlrepo import fetch_ucirepo


# the dataloader can load the original size data, and the reduced size data
def dataloader(file_path):
  data = np.load(file_path)
  X_train, y_train, X_validation, y_validation, X_test, y_test = (data['X_train'],data['y_train'],data['X_validation'],data['y_validation'],data['X_test'],data['y_test'])

  return X_train, X_validation, X_test, y_train, y_validation, y_test


# --- Preprocessing: download CTG from UCI, build feature matrix, split, save ---
# Save to your own Google Drive (not the shared drive)
file_path = '/content/drive/MyDrive/CTG_original_data.npz'

if not os.path.exists(file_path):
  print("Downloading Cardiotocography dataset from UCI...")
  ctg = fetch_ucirepo(id=193)

  X_raw = ctg.data.features.copy()                             # (2126, 21)
  y_raw = ctg.data.targets['NSP'].values.astype(np.int64) - 1  # 1/2/3 -> 0/1/2

  # Separate the categorical Tendency feature (values: -1, 0, +1)
  tendency = X_raw['Tendency'].values
  X_cont   = X_raw.drop(columns=['Tendency']).values.astype(np.float64)  # (2126, 20)

  # Normalize all 20 continuous features to [0, 1]
  scaler = MinMaxScaler()
  X_cont = scaler.fit_transform(X_cont)

  # One-hot encode Tendency: -1 -> [1,0,0]  |  0 -> [0,1,0]  |  +1 -> [0,0,1]
  # Columns 20-22 of the final feature matrix
  tendency_onehot = np.zeros((len(tendency), 3), dtype=np.float64)
  for i, t in enumerate(tendency):
    if   t == -1: tendency_onehot[i, 0] = 1.0
    elif t ==  0: tendency_onehot[i, 1] = 1.0
    else:         tendency_onehot[i, 2] = 1.0

  # Final feature matrix: [20 continuous | 3 one-hot Tendency] = 23 features total
  X_all = np.concatenate([X_cont, tendency_onehot], axis=1)  # (2126, 23)

  # Shuffle then split 60 / 20 / 20 -> train / validation / test
  R    = np.random.RandomState(random_state)
  perm = R.permutation(len(X_all))
  X_all, y_raw = X_all[perm], y_raw[perm]

  n       = len(X_all)
  n_test  = int(0.2 * n)
  n_val   = int(0.2 * n)
  X_test_raw,  y_test_raw  = X_all[:n_test],             y_raw[:n_test]
  X_val_raw,   y_val_raw   = X_all[n_test:n_test+n_val], y_raw[n_test:n_test+n_val]
  X_train_raw, y_train_raw = X_all[n_test+n_val:],       y_raw[n_test+n_val:]

  np.savez(file_path,
           X_train=X_train_raw,      y_train=y_train_raw,
           X_validation=X_val_raw,   y_validation=y_val_raw,
           X_test=X_test_raw,        y_test=y_test_raw)
  print(f"Dataset saved to {file_path}")
else:
  print(f"Found existing file: {file_path}")

# Load the dataset
# YOUR CODE HERE


# check the shape of data
# Feature layout: columns 0-19 = continuous FHR measurements, columns 20-22 = one-hot Tendency
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)
print(X_test.shape)
print(y_test.shape)


### Part 1.2: Data Reduction (2 point)
Implement data reduction based on compression ratio.

The CTG dataset has 2,126 samples. Use the given `data_compression_ratio` accordingly.

In [ ]:
# function to reduce the dataset size for our experiments
# We apply the dataset reduction to the training and validation set
# We randomly pick data points from the training and validation sets
def data_reduction(X_train, X_validation, y_train, y_validation, size_train, size_validation):

  print('Data size reduction method')

  # selecting random rows from the training set
  R = np.random.RandomState(random_state)
  rows = R.randint(X_train.shape[0], size=size_train)

  # YOUR CODE HERE

  print(f"X_train.shape: {X_train.shape}")
  print(f"y_train.shape: {y_train.shape}")

  # selecting random rows from the validation set
  rows = R.randint(X_validation.shape[0], size=size_validation)

  # YOUR CODE HERE

  return X_train, X_validation, y_train, y_validation

In [ ]:
# We can use this part to generate reduced size datasets for various data compression ratios
# setting the data compression ratio
data_compression_ratio = 2

# computing the size of the training and validation set based on the data compression ratio
size_train = int(X_train.shape[0]/data_compression_ratio)
size_validation = int(X_validation.shape[0]/data_compression_ratio)

# Reduce the dataset size with the data_compression_ratio set above
# YOUR CODE HERE

# Saving the reduced size data - we can name the data to include the data compression ratio
# Save to your own Google Drive
reduced_file_path = # YOUR CODE HERE
# e.g. '/content/drive/MyDrive/CTG_reduced_data.npz'
np.savez(reduced_file_path,
    X_train=X_train_reduced,
    y_train=y_train_reduced,
    X_validation=X_validation_reduced,
    y_validation=y_validation_reduced,
    X_test=X_test, # The test set for the reduced size data is the same as the original data
    y_test=y_test
    )
print("reduced size data successfully saved.")

In [ ]:
# load reduced data
X_train, X_validation, X_test, y_train, y_validation, y_test = dataloader(reduced_file_path)
print ("Loading complete !")

## Part 2: Synthetic Data Generation (8 points)
Objectives:
- Implement Kernel Density Estimation (KDE) for synthetic data generation
- Validate synthetic data using semantic integrity classifer
- Label synthetic data with Random Forest

### Part 2.1: KDE Implementation (4 points)
Complete the KDE sampling function

In [ ]:
from sklearn.neighbors import KernelDensity

def KDE_sample_generation (X_train, X_validation):

  # search over different bandwidth, find the best one
  bw_list = [0.5, 0.7, 1, 1.5, 2, 3]

  log_like = np.zeros((len(bw_list)))
  for i, bw in enumerate(bw_list):
    print(f"bw = {bw}")
    # create KernelDensity model with: kernel='gaussian', bandwidth=bw
    # fit using training data
    # Compute the total log-likelihood of validation data under the model (check avalible functions in sklearn.neighbors.KernelDensity), and save in log_like
    # YOUR CODE HERE

  bbw = bw_list[np.argmax(log_like)]
  print (f"Best Bandwidth: {bbw}")

  # create model with Best Bandwidth, fit
  # YOUR CODE HERE

  # sample 20000 samples from the model
  X_syn = # YOUR CODE HERE
  print(f"X_syn.shape: {X_syn.shape}")

  # Post-process the one-hot Tendency feature (columns 20-22):
  # KDE generates continuous values; binarize to the winning category.
  for i in range(X_syn.shape[0]):
    tendency_sample = X_syn[i, 20:23]
    max_index = 20 + np.argmax(tendency_sample)

    for j in range(20, 23):
      if j == max_index:
        X_syn[i, j] = 1
      else:
        X_syn[i, j] = 0

  return X_syn

### Part 2.2: Semantic integrity classifier
Define the semantic integrity classifier. For the CTG dataset, this classifier predicts the **Tendency** category (histogram asymmetry: left / symmetric / right) from the 20 continuous FHR measurements, then filters out synthetic records where the KDE-generated Tendency is inconsistent with the surrounding signal.

Because the CTG dataset is small, this cell runs in under a minute — you should run it.

In [ ]:
def softmax(vector):
  e = np.exp(vector)
  return (e / sum(e))

# semantic integrity classifier predicts the value of one categorical feature with respect to continuous features of the data
# In this case, we have one categorical feature: Tendency (FHR histogram asymmetry)
# The categorical feature is one-hot encoded in columns 20-21-22
def semantic_integrity_classifier(X_syn, X_train, X_validation):

  X_train_total = np.concatenate((X_train, X_validation))

  # indices for the continuous columns for this dataset (all 20 FHR measurement features)
  indices = np.array([list(np.arange(0, 20))]).reshape(-1)
  print(indices)


  # finding the label for each data instance; the label is the value of the categorical column
  # in this case the one-hot encoding of the Tendency feature is in columns 20-21-22.
  label = np.zeros((X_train_total.shape[0]))
  for i in range(X_train_total.shape[0]):
    tendency_sample = X_train_total[i, 20:23]
    tendency_sample = softmax(tendency_sample)
    index = np.argmax(tendency_sample)
    label[i] = index

  # label train shows the value of the categorical feature in the training data
  label_train = np.zeros((X_train.shape[0]))
  for i in range(X_train.shape[0]):
    tendency_sample = X_train[i, 20:23]
    tendency_sample = softmax(tendency_sample)
    index = np.argmax(tendency_sample)
    label_train[i] = index

  # label validation shows the value of the categorical feature in the validation data
  label_validation = np.zeros((X_validation.shape[0]))
  for i in range(X_validation.shape[0]):
    tendency_sample = X_validation[i, 20:23]
    tendency_sample = softmax(tendency_sample)
    index = np.argmax(tendency_sample)
    label_validation[i] = index

  # Semantic integrity classifier

  clf = RandomForestClassifier()

  # computing the train and validation accuracy of the semantic integrity classifier
  clf.fit(X_train[:, indices], label_train)
  y_pred_train = clf.predict(X_train[:, indices])
  train_acc = accuracy_score(label_train, y_pred_train)
  print('Train accuracy: {})'.format(train_acc))

  # Predict the response for validation set
  y_pred_validation = clf.predict(X_validation[:, indices])
  val_acc = accuracy_score(label_validation, y_pred_validation)
  print('Val accuracy: {})'.format(val_acc))

  # Using all the data to train the final classifier
  clf.fit(X_train_total[:, indices], label)
  X_syn_final = np.zeros((1, X_syn.shape[1]))
  print("Shape of X_syn: ", X_syn.shape)
  print("Shape of X_syn_final: ", X_syn_final.shape)


  # we predict the value of the categorical feature of the synthetic data with respect to continuous features
  # if not match, we disregard that synthetic data record
  for i in range(X_syn.shape[0]):
    predict = clf.predict(X_syn[i, indices].reshape(1, -1))

    index = np.argmax(softmax(X_syn[i, 20:23]))

    # if the Tendency value is semantically correct, we keep that synthetic data point
    if predict == index:
      for j in range(20, 23):
        if j == index+20:
          X_syn[i, j] = 1
        else:
          X_syn[i, j] = 0

      X_syn_final = np.concatenate((X_syn_final, X_syn[i, :].reshape(1,-1)), axis= 0)

  np.delete(X_syn_final, 0, 0)
  print ("Shape of final X_syn: ", X_syn_final.shape)

  return X_syn_final

#### Use above functions to generate synthetic data (1 point)

In [ ]:
# generating the synthetic data using KDE. This may take several minutes, please be patient
# YOUR CODE HERE

In [ ]:
# Use the semantic integrity classifier to filter the synthetic data.
# The CTG dataset is small enough that this runs in a few minutes on Colab (~2-5 minutes) — please run it.
# Save X_syn_final to your Drive so it can be reloaded in Part 2.3.
# YOUR CODE HERE
# Hint: use the existing function semantic_integrity_classifier and then use np.save to save it to '/content/drive/MyDrive/X_syn_CTG.npy'

### Part 2.3: Synthetic data labeling (4 points)
- Implement Random Forest for synthetic data labeling
- Load the synthetic CTG data you saved in Part 2.2 (or the instructor-provided file from Google Drive)

#### Search for best max_depth (2 points)

In [ ]:
# load synthetic data (saved by you at the end of Part 2.2)
X_syn = np.load('/content/drive/MyDrive/X_syn_CTG.npy')

In [ ]:
# Using the Random forest model to label the synthetic data
validation_accs = []
train_accs = []

# go over different max_depth of Random forest
for i in range (1, 25):
  print(f"training Random forest max_depth={i}")
  clf = RandomForestClassifier(n_estimators=350, max_depth=i, criterion='gini', random_state=random_state)

  # fit the model, and save training accuracy in train_accs, validation accuracy in validation_accs
  # YOUR CODE HERE

In [ ]:
plt.title('Training and validation accuracy vs. Tree depth')
plt.plot(train_accs, label='Training accuracy')
plt.plot(validation_accs, label='Validation accuracy')
plt.legend()
plt.show()

best_tree_id = # YOUR CODE HERE
best_tree_depth = best_tree_id + 1
print(f"Baseline Random Forest depth={best_tree_depth}")
print(f"train acc={train_accs[best_tree_id]}")
print(f"validation acc={validation_accs[best_tree_id]}")

#### Which max_depth is the best? Explain the reason. (1 point)
depth=YOUR ANSWER HERE. Reason: the best depth is the one that achieves the highest validation accuracy without overfitting to the training set.

#### fit a Random Forset with the best max_depth, and use it to label synthetic data (1 point)

In [ ]:
# after finding the best depth, we will retrain a random forset with both training and validation data

X_train_total = np.concatenate((X_train, X_validation), axis=0)
y_train_total = np.concatenate((y_train, y_validation), axis=0)

clf = RandomForestClassifier(n_estimators=350, max_depth=best_tree_depth, criterion='gini', random_state=random_state)
# fit the model with total data, and report training accuracy
# YOUR CODE HERE
print(f"train acc={train_acc}")

# label the synthetic data with current model
y_syn = # YOUR CODE HERE
print(f"y_syn: {y_syn[:10]}")
np.save('/content/drive/MyDrive/y_syn_CTG.npy', y_syn)

## Part 3: Model Training (8 points)
Objective:
- Build and train a neural network to classify CTG fetal heart rate patterns (Normal / Suspect / Pathologic)
- Compare baseline training on reduced real data vs. Schema A (synthetic pre-training + real fine-tuning)

Note that training may take a few minutes to finish. You could train for fewer epochs when debugging.

### Part 3.1: Neural Network Architecture (2 points)
Complete the DynamicNet class definition.

The network takes 23 input features (20 continuous CTG measurements + 3 one-hot Tendency) and classifies into 3 NSP categories.

In [ ]:
import torch.nn as nn
from torch.autograd import Variable
import torch.nn.functional as F

# DNN model definition
class DynamicNet(torch.nn.Module):
  def __init__(self, D_in, H_1=100, H_2=50, D_out=3):
    super(DynamicNet, self).__init__()
    self.input_linear = nn.Linear(D_in, H_1)
    self.middle_linear = nn.Linear(H_1, H_2)
    self.output_linear = nn.Linear(H_2, D_out)

  def forward(self, x):
    # Add F.leaky_relu as the activation function after self.input_linear and self.middle_linear.
    # input --> input_linear, activation, middle_linear, activation, output_linear --> output
    # YOUR CODE HERE


### Part 3.2: Baseline model training (3 points)
Train a DNN with reduced original data.

In [ ]:
# function to calculate test accuracy
def display_acc(X_test, y_test, model):
  correct = 0
  total = X_test.shape[0]
  with torch.no_grad():
    for i in range (X_test.shape[0]):
      x_t = torch.from_numpy(X_test[i,:]).type(torch.FloatTensor)
      x_t = Variable(x_t)
      x_t = x_t.to(device)

      y_t = torch.from_numpy(np.asarray(y_test[i])).type(torch.LongTensor)
      y_t = Variable(y_t)
      y_t = y_t.to(device)

      output = model.forward(x_t)
      np_output = (output.cpu()).numpy()
      y_pred = np.argmax(np_output)
      label = int(y_test[i])
      if (y_pred == label):
        correct += 1
  return correct/total

In [ ]:
# Set a fixed random seed for reproducibility
import random
random.seed(random_state)
np.random.seed(random_state)
torch.manual_seed(random_state)
if device == 'cuda':
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

# training the baseline DNN model with reduced original data
# this may take several minutes
print ("Baseline NN")

x = Variable(torch.from_numpy(X_train).type(torch.FloatTensor))
y = Variable(torch.from_numpy(y_train.reshape(-1)).type(torch.LongTensor))

# put x and y to device
# YOUR CODE HERE

# initialize model only with the input dimension (match the data)
# put model to device
# YOUR CODE HERE

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2 )

for t in range(1500):
  # train for 1500 epoch
  # YOUR CODE HERE

# print test accuracy
# YOUR CODE HERE


### Part 3.3: Learning Scheme A (3 points)
Implement the two-phase training process: Scheme A uses the synthetic data to pretrain the network architecture. Pretraining is followed by use of the real dataset (training set and validation set) to fine-tune.

In [ ]:
# Set a fixed random seed for reproducibility
random.seed(random_state)
np.random.seed(random_state)
torch.manual_seed(random_state)
if device == 'cuda':
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

# This method uses the synthetic data for DNN pretraining, and uses the real data for final training
print ("Schema A")

# initialize model only with the input dimension (match the data)
# put model to device
# YOUR CODE HERE

# define criterion and optimizer
# YOUR CODE HERE

# stage 1: pretraining with the synthetic data
# Loading the synthetic data
# YOUR CODE HERE

# put data to device
# YOUR CODE HERE

# train for 500 epoch
# YOUR CODE HERE


# stage 2: final training with real data
# construct real data: combine training set and validation set
# YOUR CODE HERE

# put data to device
# YOUR CODE HERE

# train for 1500 epoch
# YOUR CODE HERE

# print test accuracy
# YOUR CODE HERE


## Compare Schema A with baseline, comment on the performance difference. (1 point)
YOUR ANSWER HERE. Reason: (hint — discuss what role the synthetic CTG pre-training plays and why it helps or does not help when real labeled data is scarce.)